In [1]:
import json
import logging

from rag_agent_EC2 import RAGAgent

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [3]:
VECTOR_BUCKET = "bhitech-minelogx-poc-legislation-documents-vector"
INDEX_NAME = "bhitech-minelogx-chunkspdfs"
AWS_REGION = "us-east-1"

LLM_ENDPOINT = "http://ec2-98-81-228-187.compute-1.amazonaws.com:11434"
LLM_MODEL = "qwen3:8b"
EMBEDDINGS_ENDPOINT = "http://ec2-3-208-23-94.compute-1.amazonaws.com:11434"
EMBEDDINGS_MODEL = "mxbai-embed-large"

In [4]:
def print_response(label: str, raw_response: str) -> None:
    """Parse the JSON response and print the most relevant fields."""
    resp = json.loads(raw_response)

    print("\n" + "=" * 70)
    print(f"QUERY : {label}")
    print("=" * 70)

    print(f"\n[SUCCESS]  {resp['success']}")
    print(f"[QUERY]    Original : {resp['query']['original']}")
    print(f"           Optimized: {resp['query']['optimized']}")

    print(f"\n[ANSWER]\n{resp['answer']}")

    print(f"\n[SOURCES] ({resp['retrieval']['document_count']} documents used)")
    for doc in resp["retrieval"]["documents"]:
        print(
            f"  #{doc['rank']}  score={doc['similarity_score']:.4f}  "
            f"{doc['source_key'].split('/')[-1]}  "
            f"pages {(doc['start_page'] or 0) + 1}–{doc['end_page'] or '?'}"
        )
        print(f"       preview: {doc['text_preview'][:120].strip()}...")

    print(
        f"\n[PERFORMANCE]  LLM: {resp['performance']['llm_elapsed_s']}s  "
        f"Embed: {resp['performance']['embedding_elapsed_s']}s"
    )
    print(
        f"[TURN]  {resp['conversation']['turn']}  "
        f"| history: {resp['conversation']['history_turns']} turns stored"
    )

In [5]:
agent = RAGAgent(
    vector_bucket_name=VECTOR_BUCKET,
    index_name=INDEX_NAME,
    region=AWS_REGION,
    llm_endpoint=LLM_ENDPOINT,
    llm_model=LLM_MODEL,
    embeddings_endpoint=EMBEDDINGS_ENDPOINT,
    embeddings_model=EMBEDDINGS_MODEL,
    top_k=5,
    top_n=3,
)

2026-06-03 22:09:09  INFO      botocore.credentials  Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [10]:
print("\n\n" + "#" * 70)
print("# SCENARIO 1 — SIMPLE RETRIEVAL")
print("#" * 70)

q1 = "What is the purpose of the Mine Health and Safety Act of 1996?"
raw_answer = agent.chat(q1)
raw_answer

2026-06-03 22:19:20  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What is the purpose of the Mine Health and Safety Act of 1996?'




######################################################################
# SCENARIO 1 — SIMPLE RETRIEVAL
######################################################################


2026-06-03 22:19:23  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'What is the purpose of the Mine Health and Safety Act of 1996?'
2026-06-03 22:19:25  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:19:25  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking


'{\n  "success": true,\n  "query": {\n    "original": "What is the purpose of the Mine Health and Safety Act of 1996?",\n    "optimized": "What is the purpose of the Mine Health and Safety Act of 1996?"\n  },\n  "answer": "The provided context does not mention the Mine Health and Safety Act of 1996 or its purpose. The retrieved documents focus on Directive 2014/34/EU and technical requirements for mining equipment, but they do not reference national legislation such as the Mine Health and Safety Act of 1996. **CANNOT BE DETERMINED** based on the available documents.",\n  "retrieval": {\n    "vector_bucket": "bhitech-minelogx-poc-legislation-documents-vector",\n    "index": "bhitech-minelogx-chunkspdfs",\n    "top_k": 5,\n    "top_n": 3,\n    "document_count": 3,\n    "documents": [\n      {\n        "rank": 1,\n        "id": "celex-32014l0034-en-txt-chunk-9-sub-7",\n        "similarity_score": 0.647035,\n        "source_key": "docs/CELEX_32014L0034_EN_TXT.pdf",\n        "start_page": 1

In [11]:
print_response(q1, raw_answer)


QUERY : What is the purpose of the Mine Health and Safety Act of 1996?

[SUCCESS]  True
[QUERY]    Original : What is the purpose of the Mine Health and Safety Act of 1996?
           Optimized: What is the purpose of the Mine Health and Safety Act of 1996?

[ANSWER]
The provided context does not mention the Mine Health and Safety Act of 1996 or its purpose. The retrieved documents focus on Directive 2014/34/EU and technical requirements for mining equipment, but they do not reference national legislation such as the Mine Health and Safety Act of 1996. **CANNOT BE DETERMINED** based on the available documents.

[SOURCES] (3 documents used)
  #1  score=0.6470  CELEX_32014L0034_EN_TXT.pdf  pages 19–20
       preview: s category is intended for use in underground parts of mines as well as those parts of surface 
installations of such mi...
  #2  score=0.6423  CELEX_32014L0034_EN_TXT.pdf  pages 19–20
       preview: ment in this category is intended for use in underground parts of mines a

In [6]:
print("\n\n" + "#" * 70)
print("# SCENARIO 1 — SIMPLE RETRIEVAL")
print("#" * 70)

q1 = "What is the purpose of the Mine Health and Safety Act of 1996?"
raw_answer = agent.chat(q1)
print_response(q1, agent.chat(q1))

q2 = "What rights do employees have regarding dangerous working conditions?"
print_response(q2, agent.chat(q2))

q3 = "What institutions does the Act establish for health and safety oversight?"
print_response(q3, agent.chat(q3))

2026-06-03 22:09:10  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What is the purpose of the Mine Health and Safety Act of 1996?'




######################################################################
# SCENARIO 1 — SIMPLE RETRIEVAL
######################################################################


2026-06-03 22:09:12  INFO      rag_agent_EC2  [RAGAgent] Optimized query: '"Mine Health and Safety Act 1996 purpose"'
2026-06-03 22:09:15  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:09:15  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking
2026-06-03 22:09:19  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What rights do employees have regarding dangerous working conditions?'



QUERY : What is the purpose of the Mine Health and Safety Act of 1996?

[SUCCESS]  True
[QUERY]    Original : What is the purpose of the Mine Health and Safety Act of 1996?
           Optimized: "Mine Health and Safety Act 1996 purpose"

[ANSWER]
The purpose of the Mine Health and Safety Act of 1996 is not addressed in the retrieved documents. The provided texts reference regulations related to equipment protection against explosive atmospheres (e.g., Directive 94/9/EC) and general health and safety requirements for mining equipment, but they do not mention the specific Mine Health and Safety Act of 1996 or its legislative intent. 

**Compliance Verdict:** CANNOT BE DETERMINED  
**Reason:** The retrieved documents do not cover the Mine Health and Safety Act of 1996 or its purpose.

[SOURCES] (3 documents used)
  #1  score=0.6691  CELEX_32014L0034_EN_TXT.pdf  pages 19–20
       preview: s category is intended for use in underground parts of mines as well as those parts of surface 
inst

2026-06-03 22:09:22  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'What rights do employees have regarding dangerous working conditions?'
2026-06-03 22:09:26  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:09:26  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking
2026-06-03 22:09:31  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What institutions does the Act establish for health and safety oversight?'



QUERY : What rights do employees have regarding dangerous working conditions?

[SUCCESS]  True
[QUERY]    Original : What rights do employees have regarding dangerous working conditions?
           Optimized: What rights do employees have regarding dangerous working conditions?

[ANSWER]
**Answer:**  
The retrieved documents do not explicitly address employees' specific rights regarding dangerous working conditions. The texts focus on regulatory obligations for manufacturers and general health and safety requirements for equipment, but they do not outline employee rights such as the right to a safe workplace, the right to report hazards, or entitlements to compensation for work-related injuries.  

**Compliance Verdict:** CANNOT BE DETERMINED  
**Reason:** The provided documents do not cover employee rights regarding dangerous working conditions. They emphasize regulatory requirements for product safety and hazard protection but do not specify rights attributable to employees.  

**Re

2026-06-03 22:09:34  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'What institutions does the Act establish for health and safety oversight?'
2026-06-03 22:09:37  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:09:37  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking



QUERY : What institutions does the Act establish for health and safety oversight?

[SUCCESS]  True
[QUERY]    Original : What institutions does the Act establish for health and safety oversight?
           Optimized: What institutions does the Act establish for health and safety oversight?

[ANSWER]
**Answer:**  
The retrieved documents do not explicitly identify specific institutions established by the Act for health and safety oversight. The texts focus on regulatory requirements for equipment safety, harmonized standards, and manufacturer obligations (e.g., Article 4, Document 2; Annex II, Document 1), but they do not mention statutory bodies, regulatory agencies, or oversight institutions created by the Act.  

**Compliance Verdict:** CANNOT BE DETERMINED  
**Reason:** The provided documents do not cover the establishment of institutions for health and safety oversight. They emphasize technical and procedural requirements for compliance rather than governance structures or regulat

In [7]:
print("\n\n" + "#" * 70)
print("# SCENARIO 4 — OUT OF SCOPE (expect CANNOT BE DETERMINED)")
print("#" * 70)

agent.reset_history()

q7 = (
    "What is the maximum permissible noise exposure level in decibels "
    "for a worker inside a mine shaft during an 8-hour shift?"
)
print_response(q7, agent.chat(q7))

print("\n\nAll test scenarios complete.")

2026-06-03 22:09:46  INFO      rag_agent_EC2  [RAGAgent] Conversation history cleared.
2026-06-03 22:09:46  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What is the maximum permissible noise exposure level in decibels for a worker inside a mine shaft during an 8-hour shift?'




######################################################################
# SCENARIO 4 — OUT OF SCOPE (expect CANNOT BE DETERMINED)
######################################################################


2026-06-03 22:09:48  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'What is the maximum permissible noise exposure level in decibels for a worker inside a mine shaft during an 8-hour shift?'
2026-06-03 22:09:56  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:09:56  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking



QUERY : What is the maximum permissible noise exposure level in decibels for a worker inside a mine shaft during an 8-hour shift?

[SUCCESS]  True
[QUERY]    Original : What is the maximum permissible noise exposure level in decibels for a worker inside a mine shaft during an 8-hour shift?
           Optimized: What is the maximum permissible noise exposure level in decibels for a worker inside a mine shaft during an 8-hour shift?

[ANSWER]
The context provided does not contain any information regarding noise exposure levels, permissible decibel limits, or related regulatory requirements for mining environments. The referenced documents (pages 19–20 of CELEX_32014L0034_EN_TXT.pdf) focus exclusively on equipment classification, protection mechanisms, and operational parameters for mines endangered by firedamp or combustible dust. They do not address noise exposure thresholds or permissible levels for workers.

**Compliance Determination:**  
CANNOT BE DETERMINED  
**Reason:** The retri

In [8]:
# ---------------------------------------------------------------------------
# Scenario 3: Multi-turn conversation -- Directive 2014/34/EU
# Verifies that history is preserved and pronouns are resolved correctly
# ---------------------------------------------------------------------------

print("\n\n" + "#" * 70)
print("# SCENARIO 3 -- MULTI-TURN CONVERSATION (Directive 2014/34/EU)")
print("#" * 70)

agent.reset_history()

t1 = "What are the obligations of manufacturers under Directive 2014/34/EU?"
print_response("Turn 1: " + t1, agent.chat(t1))

# "they" and "it" rely on history to resolve subject
t2 = "How long do they need to keep the technical documentation for?"
print_response("Turn 2 (pronoun resolution): " + t2, agent.chat(t2))

# Drilling deeper into the same topic
t3 = "And what must they do if they believe a product they placed on the market is not compliant?"
print_response("Turn 3 (follow-up): " + t3, agent.chat(t3))

2026-06-03 22:14:38  INFO      rag_agent_EC2  [RAGAgent] Conversation history cleared.
2026-06-03 22:14:38  INFO      rag_agent_EC2  [RAGAgent] Processing: 'What are the obligations of manufacturers under Directive 2014/34/EU?'




######################################################################
# SCENARIO 3 -- MULTI-TURN CONVERSATION (Directive 2014/34/EU)
######################################################################


2026-06-03 22:14:41  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'What are the obligations of manufacturers under Directive 2014/34/EU?'
2026-06-03 22:14:42  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:14:42  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking
2026-06-03 22:14:54  INFO      rag_agent_EC2  [RAGAgent] Processing: 'How long do they need to keep the technical documentation for?'



QUERY : Turn 1: What are the obligations of manufacturers under Directive 2014/34/EU?

[SUCCESS]  True
[QUERY]    Original : What are the obligations of manufacturers under Directive 2014/34/EU?
           Optimized: What are the obligations of manufacturers under Directive 2014/34/EU?

[ANSWER]
The obligations of manufacturers under Directive 2014/34/EU are as follows:  

1. **Compliance with Directive Requirements**  
   - **Value/Statement**: Manufacturers must ensure their products comply with the directive when placing them on the market or using them in their territory.  
   - **Regulatory Limit/Requirement**: Article 6(1) of the directive states that manufacturers must ensure compliance with the directive’s requirements.  
   - **Compliance Verdict**: COMPLIES.  
   - **Source**: Directive 2014/34/EU, Article 6(1) (Document 2, pages 7–8).  
   - **Observation**: This obligation applies to both placing products on the market and their use within the territory.  

2. **Extended O

2026-06-03 22:14:57  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'How long do they need to keep the technical documentation for?'
2026-06-03 22:15:00  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:15:00  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking
2026-06-03 22:15:06  INFO      rag_agent_EC2  [RAGAgent] Processing: 'And what must they do if they believe a product they placed on the market is not compliant?'



QUERY : Turn 2 (pronoun resolution): How long do they need to keep the technical documentation for?

[SUCCESS]  True
[QUERY]    Original : How long do they need to keep the technical documentation for?
           Optimized: How long do they need to keep the technical documentation for?

[ANSWER]
1. **Value/Statement**: The technical documentation must be kept for a specific period.  
2. **Regulatory Limit/Requirement**: Article 2.2 of Document 1 (pages 41–42) states that manufacturers must keep technical documentation "at the disposal of the relevant national authorities for 10 years after the product has been placed on the market." Document 2 (pages 7–8) reinforces this requirement, specifying the same 10-year retention period.  
3. **Compliance Verdict**: COMPLIES.  
4. **Source**: Directive 2014/34/EU, Article 2.2 (Document 1, pages 41–42) and Article 4 (Document 2, pages 7–8).  
5. **Observation**: The retention period applies to both technical documentation and the EU declaration

2026-06-03 22:15:09  INFO      rag_agent_EC2  [RAGAgent] Optimized query: 'And what must they do if they believe a product they placed on the market is not compliant?'
2026-06-03 22:15:09  INFO      rag_agent_EC2  [RAGAgent] Retrieved 5 raw documents
2026-06-03 22:15:09  INFO      rag_agent_EC2  [RAGAgent] Kept 3 documents after reranking



QUERY : Turn 3 (follow-up): And what must they do if they believe a product they placed on the market is not compliant?

[SUCCESS]  True
[QUERY]    Original : And what must they do if they believe a product they placed on the market is not compliant?
           Optimized: And what must they do if they believe a product they placed on the market is not compliant?

[ANSWER]
1. **Value/Statement**: Manufacturers must take specific actions if they believe a product they placed on the market is non-compliant.  
2. **Regulatory Limit/Requirement**: Document 1 (pages 9–10) states: *"who consider or have reason to believe that a product which they have placed on the market is not in conformity with this Directive shall immediately take the corrective measures necessary to bring that product into conformity, to withdraw it or recall it, if appropriate."*  
3. **Compliance Verdict**: COMPLIES.  
4. **Source**: Directive 2014/34/EU, Article 6(1) (Document 1, pages 9–10).  
5. **Observation**: Th